## Reading 32-2 - Undefined Behavior and the Compiler as Your First Analyzer

By the end of this reading, you will learn:
<ol>
    <li>What <i>undefined behavior</i> is in C, and why it is the most common way C programs fail silently</li>
    <li>Why the same program can produce different answers with different compiler flags</li>
    <li>How the compiler flags we have used all semester act as your first line of defense</li>
    <li>Three new tools: <code>-fanalyzer</code>, AddressSanitizer, and UndefinedBehaviorSanitizer</li>
</ol>

### Undefined Behavior

The C standard divides "things the language does not fully specify" into two groups:

<ul>
    <li><b>Implementation-defined behavior:</b> The compiler must pick an answer and document it.</li> 
    <ul><li>Example: whether <code>char</code> is signed or unsigned on a given machine.</li></ul>
    <li><b>Undefined behavior (UB):</b> The C standard places <i>no requirements at all</i> on what happens. The program might work, crash, or print garbage.</li>
</ul>

Common sources of undefined behavior include:
<ul>
    <li>Signed integer overflow</li>
    <li>Reading or writing past the end of an array</li>
    <li>Reading a variable before it has been given a value</li>
    <li>Using memory after it has been freed</li>
</ul>

<b>The key idea:</b> The compiler is allowed to assume that undefined behavior <i>never happens</i>, and it optimizes your code based on that assumption. This means a bug can be invisible at one optimization level and appear at another. A test that passes on your machine can fail on the flight computer.

### Example 1 - Signed Overflow - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec32/signed_overflow.c">signed_overflow.c</a>

This function asks whether adding 1 to <code>x</code> makes it <i>smaller</i>, which can only happen if the value wraps around:

    #include <stdio.h>
    #include <stdlib.h>
    #include <limits.h>

    /* Is x + 1 smaller than x? Mathematically, only if x + 1 "wrapped around". */
    int wraps(int x)
    {
        return x + 1 < x;
    }

    int main(void)
    {
        int x = INT_MAX;
        fprintf(stdout, "x = %d, wraps(x) = %d\n", x, wraps(x));
        return 0;
    }


From our work with two's complement, we would expect <code>INT_MAX + 1</code> to wrap to <code>INT_MIN</code>, so <code>wraps(INT_MAX)</code> should return 1. Here is what actually happens:

    $ ./signed_overflow
    x = 2147483647, wraps(x) = 0

The compiler reasoned: "Signed overflow is undefined, so <code>x + 1</code> can never be less than <code>x</code>. I will replace this whole expression with 0." It never performed the addition. No warning was printed.

Now compile with the <b>UndefinedBehaviorSanitizer</b>, which adds run-time checks for undefined behavior:

    $ gcc -std=c2x -O2 -g -Wall -Wextra -Wconversion -Wshadow -Werror -fsanitize=undefined signed_overflow.c -o signed_overflow
    $ ./signed_overflow
    signed_overflow.c:8:11: runtime error: signed integer overflow: 2147483647 + 1 cannot be represented in type 'int'
    x = 2147483647, wraps(x) = 1

The sanitizer catches the overflow while the program runs, and tells you the file and line.

### Example 2 - Integer Promotion - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec32/promotion.c">promotion.c</a>

Before C does arithmetic on small integer types, it <i>promotes</i> them to <code>int</code>. For example, <code>uint16_t</code> is an unsigned 16-bit integer.

    #include <stdio.h>
    #include <stdint.h>

    int main(void)
    {
        uint16_t a = 65535;
        uint16_t b = 1;

        uint32_t sum32 = a + b;	/* a and b are promoted to int before adding */
        uint16_t sum16 = a + b;	/* the int result is truncated on the store */

        fprintf(stdout, "sum32 = %u\n", sum32);
        fprintf(stdout, "sum16 = %u\n", sum16);
        return 0;
    }

The result:

    $ ./promotion
    sum32 = 65536
    sum16 = 0

The same expression, <code>a + b</code>, gives two different answers depending on where you store it. In <code>sum16</code>, the value 65536 needs 17 bits, so the 17th bit is simply dropped.

Here is the surprising part: this compiles cleanly <i>even with</i> <code>-Wconversion -Werror</code>. Compiler warnings are valuable, but they do not catch everything. This is exactly the kind of error that sank Ariane 5: a value that no longer fits in the type you chose.

### Example 3 - Writing Past the End of an Array - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec32/off_by_one.c">off_by_one.c</a>

An array of 8 integers has valid indices 0 through 7. This loop uses <code>&lt;=</code> instead of <code>&lt;</code>:

    int buf[8];
    for (int i = 0; i <= 8; i++) {
    	buf[i] = 0;
    }

This time, our course flags catch it:

    off_by_one.c:7:24: error: iteration 8 invokes undefined behavior [-Werror=aggressive-loop-optimizations]
    off_by_one.c:7:24: error: '__builtin_memset' forming offset [32, 35] is out of the bounds [0, 32] of object 'buf' with type 'int[8]' [-Werror=array-bounds=]

This is why we have used <code>-Wall -Wextra -Werror</code> since the first homework. <b>The compiler is your first static analyzer.</b> It read your code without running it and proved that iteration 8 writes outside the array.

But the compiler can only catch what it can see. In <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec32/off_by_one_runtime.c">off_by_one_runtime.c</a>, the loop bound comes from the command line:

    #include <stdio.h>
    #include <stdlib.h>

    #define BUF_SIZE 8

    /* Fill the first count + 1 entries of the buffer with zeros. */
    void zero_buffer(int buf[], int count)
    {
        for (int i = 0; i <= count; i++) {
            buf[i] = 0;
        }
    }

    int main(int argc, char *argv[])
    {
        int buf[BUF_SIZE] = { 1, 2, 3, 4, 5, 6, 7, 8 };
        int guard = 42;		/* sits near buf on the stack */

        if (argc != 2) {
            fprintf(stderr, "Usage: %s <count>\n", argv[0]);
            exit(-1);
        }

        int count = (int)strtol(argv[1], NULL, 10);
        zero_buffer(buf, count);

        fprintf(stdout, "buf[0] = %d, guard = %d\n", buf[0], guard);
        return 0;
    }

It compiles cleanly, and running it with <code>8</code> writes past the end of the array <i>with no visible error</i>:

    $ ./off_by_one_runtime 8
    buf[0] = 0, guard = 42

Now compile with <b>AddressSanitizer</b>, which watches every memory access while the program runs:

    $ gcc -std=c2x -O2 -g -Wall -Wextra -Wconversion -Wshadow -Werror -fsanitize=address off_by_one_runtime.c -o off_by_one_runtime
    $ ./off_by_one_runtime 8
    ==326==ERROR: AddressSanitizer: stack-buffer-overflow on address 0x7f44c6c00040
    WRITE of size 4 at 0x7f44c6c00040 thread T0
        #0 0x55bbbb2ad520 in zero_buffer off_by_one_runtime.c:10
        #1 0x55bbbb2ad520 in main off_by_one_runtime.c:25

AddressSanitizer stops the program at the exact write that went out of bounds, and gives the line number and the chain of function calls that led there.

### Example 4 - Reading an Uninitialized Variable - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec32/uninit.c">uninit.c</a>

If the program is run with no command-line arguments, <code>mode</code> is never given a value:

    int mode;
    if (argc > 1) {
    	mode = 1;
    }
    fprintf(stdout, "mode = %d\n", mode);

Watch what happens when we change <i>only</i> the optimization level:

    $ gcc ... -O0 uninit.c -o uninit && ./uninit
    mode = 32767
    $ gcc ... -O2 uninit.c -o uninit && ./uninit
    mode = 1

Same source code. Same input. Two different answers. At <code>-O0</code>, we print whatever was left over on the stack, so your number will probably differ from the one shown here, and may change from run to run. At <code>-O2</code>, the compiler decided that since reading <code>mode</code> before assignment is undefined, it may as well assume <code>mode</code> is 1.

This is the scariest kind of bug in flight software: the version you tested and the version you flew can behave differently. You already have a tool that catches it. Running the <code>-O0</code> version under <code>valgrind</code> reports:

    ==348== Conditional jump or move depends on uninitialised value(s)
    ==348==    by 0x109189: main (uninit.c:10)

### Example 5 - Following Every Path with <code>-fanalyzer</code> - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec32/leaky_reading.c">leaky_reading.c</a>

The sanitizers only check the paths your program actually runs. If the test never takes a branch, the sanitizers never see it.

GCC's static analyzer, enabled with <code>-fanalyzer</code>, takes a different approach: it explores <i>every</i> path through a function without running the code. Consider this function, which averages temperature readings from a file:

    double *readings = (double *)malloc(NUM_READINGS * sizeof(double));

    FILE *fp = fopen(filename, "r");
    if (fp == NULL) {
    	return -1.0;	/* early exit: what happened to readings? */
    }

With a valid file, the program works and prints the right answer. With <code>-fanalyzer</code>:

    leaky_reading.c:12:12: error: leak of 'readings' [CWE-401] [-Werror=analyzer-malloc-leak]
       (1) allocated here
       (2) following 'true' branch (when 'fp' is NULL)...
       (4) 'readings' leaks here; was allocated at (1)

The analyzer walked down the path where the file does not exist, a path our test never took, and found that the memory is never freed. On a spacecraft that retries a failed file read once per second, this leak would eventually exhaust memory.

The <code>CWE-401</code> label refers to the Common Weakness Enumeration, an industry catalog of known software weaknesses.

### Layers of Defense

No single tool catches everything. Each one sees a different part of the problem:

<table>
    <tr><th>Tool</th><th>When it runs</th><th>Strength</th><th>Blind spot</th></tr>
    <tr><td>Compiler warnings (<code>-Wall -Wextra -Wconversion -Werror</code>)</td><td>Every compile</td><td>Fast, always on</td><td>Only what it can prove locally (missed Examples 1, 2, and 4)</td></tr>
    <tr><td><code>-fanalyzer</code></td><td>Compile time</td><td>Checks paths your tests never run</td><td>Slower; can report false alarms</td></tr>
    <tr><td>AddressSanitizer (<code>-fsanitize=address</code>)</td><td>Run time</td><td>Pinpoints bad memory accesses</td><td>Only sees paths that actually execute</td></tr>
    <tr><td>UndefinedBehaviorSanitizer (<code>-fsanitize=undefined</code>)</td><td>Run time</td><td>Catches overflow and other UB as it happens</td><td>Only sees paths that actually execute</td></tr>
    <tr><td><code>valgrind</code></td><td>Run time</td><td>Leaks and uninitialized reads, no recompile needed</td><td>Slow; only sees paths that execute</td></tr>
</table>

And one more column that belongs on every row: <b>none of these tools would have saved the Mars Climate Orbiter.</b> Mixing up pound-force seconds and newton-seconds is perfectly legal, memory-safe C. Catching that kind of error takes requirements, reviews, and engineers who ask what the numbers mean.

### <font color = "red">Class Introduction Question #4 - What is undefined behavior, and why can a program with undefined behavior produce different results at <code>-O0</code> and <code>-O2</code>?</font>

### <font color = "red">Class Introduction Question #5 - Example 3 and <code>off_by_one_runtime.c</code> contain the same bug. Why did the compiler catch one but not the other, and which tool caught the second?</font>

### <font color = "red">Class Introduction Question #6 - What can <code>-fanalyzer</code> find that AddressSanitizer cannot, and what can AddressSanitizer find that <code>-fanalyzer</code> cannot?</font>